In [1]:
import sys
from pathlib import Path
import os, json, pickle, datetime as dt

os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)

sys.path.insert(0, "..")

from src.seed import set_seed
set_seed(42)

# Data Science Project Architecture
## Getting a feel of an end-to-end data science solution

In this lab, you'll see how all the pieces of data science: data analysis, code, tooling, experiments, come together to create a complete project. You'll perform a smaller-scale demonstration of a data science project lifecycle. Of course, you have to keep in mind that "real-life" data science is highly iterative. You might be working on the same task(s) for weeks or months - this lab is not able to show that.

You'll be working with the asthma dataset located [here](https://www.kaggle.com/datasets/rabieelkharoua/asthma-disease-dataset). As always, it's preloaded for you in the `data/` directory. **Your main goal is to predict what factors lead to positive diagnosis.**

This time, I suggest you do your research into separate notebooks, not inside this one. Use one or several, as you see fit; there are no guidelines as to how many notebooks you should have, or how long (or short) they have to be.

### Problem 1. Project structure (1 point)
Create the necessary directories and structure that you'll use to put your work in. I am providing a suggestion, but you don't have to follow it.
`data/` for... data :D
`notebooks/` for your research. Feel free to move this one inside.
`src/` for Python code (which you'll need to create towards the end of the lab)
`test/` (or `tests/`) for unit tests

You may add any other structure you like. For inspiration, you can see how popular libraries handle their file structure.

Create a GitHub repo (or any other Git-based source control, but I **highly** suggest GitHub) containing your initial project structure. Don't forget to commit relatively often so you have a way to track what you've done so far and go back if something goes awry.

### Problem 2. Data Exploration (1 point)
In an appropriate notebook, load the data. Ensure its validity and start your EDA. Feel free to create any visualizations, tables, filters, etc. you see fit.

### Problem 3. Data cleaning and preprocessing (1 point)
This should be self-explanatory. In an appropriate notebook (probably different than your previous one), explore different ways to clean and preprocess the dataset.

This is still part of your research. That is, don't be afraid to _try out different approaches to the same problem_. E.g., if you have a lot of missing values, you may not know right away how to handle them. Experimenting with several approaches will give you a better indication what works well for your data and goals.

### Problem 4. Exploratory data analysis (1 point)
This step may, or may not, happen in unison with the previous one. Your goal is to understand the data distributions, relationships, useful features, maybe create visualizations and inform your data cleaning process.

### Problem 5. Feature manipulation (1 point)
Now that your data has been thoroughly cleaned (w.r.t. your goal to model diagnoses) and explored, you'll need to "play around" and prepare good features.

You don't have to think about modelling (machine learning) at this stage (although it won't do harm). Perform feature selection and feature engineering in ways that you think will be beneficial for a "mental" model of the data. Such a model consists of hypotheses that you should be able to test.

Feel free to do any sort of feature maniplulation on the data you like. Ideally, at the end of the process, you'll have a rectangular data table consisting of only (floating-point) numbers and nothing else.

### Problem 6. Data preparation and manipulation script (2 points)
So far, you should have tried lots of different ways to work with the data. Some of them should have been good, others - not so much. This is extremely valuable research, and we don't want to lose it, but now we have to think about automation.

Extract your data preprocessing and manipulation functions into one or more files in the `src/` (or similar) directory. Debug the code and ensure it's been optimized. Use vectorization and the `pandas` / `numpy` APIs as much as possible. I don't usually expect data processing scripts to create visualizations. Most often, they consist of functions which accept some dataframe(s) and return (an)other dataframe(s). Also, we usually avoid one-liners (e.g., a function which only calls a different function) unless there's a very good reason for them (e.g., it's semantically easier to understand).

Refactor the code so that it's **reusable**. Function parameters (and polymorphism) achieve a lot in terms of reusability :). Avoid hardcoding stuff. Follow the best practices in Python and the style guides. Use a linter to help you clean up your code.

### Problem 7. Documentation (1 point)
Ensure all your public-facing functions (that is, functions that are "exposed" to the user) have docstrings. Ensure they are well-documented and their purpose is clear. This is especially valuable if you're doing some advanced analysis or data manipulation. You can see various ways of creating docstrings online. There are even tools (e.g., VSCode extensions) which will help you with the docstring boilerplate.

### Problem 8. Testing (1 point)
Now that you've done the previous two problems, you have _specification_ (your documentation - it tells you what you intend to do) and _implementation_ (your well-written and refactored code - it tells you _how_ it's done). The difficult part now is to ensure these two things match.

Create unit tests for your functions. Be careful so you test _your_ code, not `pandas`'. Create hypothesis tests to validate your assumptions. Do validity checks on the input data and sanity checks on the outputs of functions. Ensure your code is well-tested. Ensure it's modular, reusable, and flexible; but most of all - that it works **correctly**. If you haven't yet (though you should have - in problem 6) - think about exceptions and exception handling.

### Problem 9. Reproducibility (1 point)
Ensure all your notebooks and scripts are not only correct, but also reproducible. List all code dependencies (probably in a `requirements.txt` file); ensure your random seeds are correct; ensure the code produces the same results when run multiple times, etc.

Do your final cleanup work. You might want to differentiate your "draft" noteoboks from your "official" ones (although I advise against that) and creat your final commits.

### * Problem 10. Above and beyond
Of course, there are many things to be done. If you have time, I advise you learn how to work with data versioning (using DVC) and data pipeline / artifact tracking (using MLFlow or a similar tool). You might also find it useful to create a "proper", advanced data pipeline where you may need to work with big files (using Dask or a similar library), or schedule and organize tasks (using data pipeline managers like Luigi or Airflow).

You might also want to do machine learning. I've deliberately stayed away from that for the purposes of the lab because it's a whole different beast, but it's a worthy challenge and it's extremely interesting.

In [2]:
try:
    from src.seed import set_seed  
except Exception:
    def set_seed(seed=42):
        import random
        os.environ["PYTHONHASHSEED"] = str(seed)
        random.seed(seed)
        try:
            import numpy as np
            np.random.seed(seed)
        except Exception:
            pass

SEED = 42

In [3]:
INPUT_CSV = "../data/processed/features_final.csv"      
MODEL_OUT = "../models/baseline_logreg.pkl"
REPORT_DIR = Path("../reports")
FIG_DIR = REPORT_DIR / "figures"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
Path("models").mkdir(exist_ok=True)

In [4]:
TARGET = "Diagnosis"
TEST_SIZE = 0.25
C = 1.0

In [5]:
df = pd.read_csv(INPUT_CSV)
assert df.columns[-1] == TARGET, f"Expected target last column: {TARGET}"
X = df.iloc[:, :-1].astype(float).to_numpy()
y = df[TARGET].astype(int).to_numpy()

In [6]:
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

In [7]:
model = LogisticRegression(
    C=float(C), solver="liblinear", penalty="l2",
    class_weight="balanced", random_state=SEED, max_iter=1000
)
model.fit(Xtr, ytr)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                   solver='liblinear')

In [8]:
p_te = model.predict_proba(Xte)[:, 1]
yhat = (p_te >= 0.5).astype(int)

In [9]:
metrics = {
    "acc": float(accuracy_score(yte, yhat)),
    "precision": float(precision_score(yte, yhat, zero_division=0)),
    "recall": float(recall_score(yte, yhat, zero_division=0)),
    "f1": float(f1_score(yte, yhat, zero_division=0)),
    "auc": float(roc_auc_score(yte, p_te)),
    "n_train": int(len(ytr)),
    "n_test": int(len(yte)),
}
print("Baseline logistic regression metrics:", json.dumps(metrics, indent=2))

Baseline logistic regression metrics: {
  "acc": 0.6120401337792643,
  "precision": 0.05726872246696035,
  "recall": 0.41935483870967744,
  "f1": 0.10077519379844961,
  "auc": 0.5189167662285943,
  "n_train": 1794,
  "n_test": 598
}


In [10]:
with open(MODEL_OUT, "wb") as f:
    pickle.dump(model, f)

In [11]:
metrics_json = REPORT_DIR / "metrics_baseline.json"
json.dump({
    "input": str(Path(INPUT_CSV).resolve()),
    "model": str(Path(MODEL_OUT).resolve()),
    "seed": SEED,
    "test_size": TEST_SIZE,
    "C": C,
    "metrics": metrics,
    "timestamp": dt.datetime.now(dt.UTC).isoformat().replace("+00:00", "Z"),
}, open(metrics_json, "w"), indent=2)

In [12]:
cm = confusion_matrix(yte, yhat)
plt.figure()
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix — Baseline")
plt.xlabel("Predicted"); plt.ylabel("Actual")
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix_baseline.png")
plt.close()

In [13]:
fpr, tpr, _ = roc_curve(yte, p_te)
plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"ROC Curve — AUC={metrics['auc']:.3f}")
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_curve_baseline.png")
plt.close()

In [14]:
log_csv = REPORT_DIR / "experiments.csv"
row = {
    "timestamp": dt.datetime.now(dt.UTC).isoformat().replace("+00:00", "Z"),
    "input": INPUT_CSV,
    "model": "logreg_liblinear_balanced",
    "seed": SEED,
    "test_size": TEST_SIZE,
    "C": C,
    **metrics,
}

In [15]:
pd.DataFrame([row]).to_csv(log_csv, mode="a", header=not log_csv.exists(), index=False)

In [16]:
print("Saved:")
print("model->", MODEL_OUT)
print("metrics->", metrics_json)
print("plots->", FIG_DIR / "confusion_matrix_baseline.png", "and", FIG_DIR / "roc_curve_baseline.png")
print("log->", log_csv)

Saved:
model-> ../models/baseline_logreg.pkl
metrics-> ..\reports\metrics_baseline.json
plots-> ..\reports\figures\confusion_matrix_baseline.png and ..\reports\figures\roc_curve_baseline.png
log-> ..\reports\experiments.csv
